# Backtesting Engine

## What is Backtesting?

**Backtesting** answers a simple question: *"If I had followed this formula's stock picks in the past, would I have made money — and would I have beaten the S&P 500?"*

Instead of trusting that the ranking formula works in theory, we test it against **real historical price data**. Every month, `python -m magic_formula run` saves the top 100 ranked stocks to the `Results/` folder. This notebook loads all of those historical snapshots, grabs what the top picks' actual prices were on those dates, and checks what those prices became 1, 3, 6, and 12 months later. It then compares those returns against **SPY** (the S&P 500 ETF) over the exact same periods.

The result is an honest, data-driven answer to: *"Does this formula beat the market?"*

---

## How to Run It

Run the five cells below **in order**, top to bottom. Each cell depends on the one above it.

| Cell | What it does | Approx. run time |
|------|-------------|-----------------|
| **1 — Imports** | Loads all required Python libraries | < 2 seconds |
| **2 — BT-Config** | Loads all `Results/stocks_*.csv` snapshots, extracts the top-N picks from each | < 5 seconds |
| **3 — BT-Prices** | Downloads adjusted closing prices for every picked stock + SPY via yfinance | 30–90 seconds |
| **4 — BT-Returns** | Calculates equal-weight portfolio return vs SPY for each snapshot × hold period; prints a summary table | < 5 seconds |
| **5 — BT-Visualize** | Generates three interactive Plotly charts | < 5 seconds |

> **Tip:** You can change `TOP_N` in BT-Config (default: 10) to test whether the top 5 or top 20 picks perform differently. Re-run BT-Returns and BT-Visualize after any change — you do **not** need to re-download prices.

---

## Understanding the Output

**Summary table** (printed by BT-Returns):

| Column | Meaning |
|--------|---------|
| `Vintages` | How many historical snapshots had enough data for that hold period |
| `Portfolio_Avg` | Average return of the top-N portfolio across all vintages |
| `SPY_Avg` | Average SPY return over the same periods |
| `Outperf_Avg` | Average edge over SPY — positive means beating the market on average |
| `Win_Rate_Pct` | % of individual snapshots where the portfolio beat SPY |
| `Best_Vintage` | Largest single-vintage outperformance |
| `Worst_Vintage` | Worst single-vintage underperformance |

**Charts** (generated by BT-Visualize):
1. **Bar charts** (one per hold period) — blue bars are the portfolio, orange bars are SPY, for every historical snapshot date. The chart title shows the win rate and average outperformance for that period.
2. **Outperformance heatmap** — one row per snapshot date, one column per hold period. Green = beat SPY, Red = missed. Empty cells are snapshots too recent to evaluate yet.
3. **Cumulative growth curve** — simulates investing $10,000 at the first snapshot and rebalancing to the new top-N picks each month. Shows the compounded result vs simply holding SPY over the same period.

---

## Important Caveats

- **No transaction costs** — brokerage fees, bid/ask spreads, and taxes are not modelled. Real returns will be slightly lower.
- **Equal weighting** — each of the top-N stocks gets the same allocation. You could weight by conviction score in a future version.
- **Survivorship bias** — tickers that were delisted or became unavailable in yfinance are silently excluded from that vintage's portfolio, which can slightly inflate portfolio returns.
- **Small sample** — `Results/` history only goes back to March 2024 (about three dozen snapshots, some irregularly spaced). That is informative but not statistically conclusive.
- **Not investment advice** — past performance of a screen says little about future results. See the disclaimer in the README.

In [ ]:
## Cell 1 — Imports
import nbformat  # import before plotly so the Jupyter renderer can find it
import yfinance as yf
import pandas as pd
import numpy as np
import glob, os, warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime as _dt

# Defensive patch: if plotly loaded before nbformat was installed (e.g., pip
# install without a kernel restart), its internal nbformat reference is None.
# Re-point it at the now-available module so fig.show() works without restarting.
import plotly.io._renderers as _plotly_renderers
if not getattr(_plotly_renderers, 'nbformat', None):
    _plotly_renderers.nbformat = nbformat

warnings.filterwarnings('ignore')

In [ ]:
## Cell 2 — BT-Config: Parameters & Load Snapshots

# ── Configurable parameters ────────────────────────────────────────────────────
TOP_N = 10            # How many top-ranked stocks to hold per snapshot
HOLD_PERIODS = {      # Label → calendar months to hold
    '1M':  1,
    '3M':  3,
    '6M':  6,
    '12M': 12,
}
MIN_VALID_TICKERS = 5  # Skip a vintage if fewer than this many tickers have price data

# ── Load all snapshot CSVs ─────────────────────────────────────────────────────
results_dir = './Results/'
csv_files = sorted(glob.glob(f'{results_dir}stocks_*.csv'))

snapshots = {}
for f in csv_files:
    date_str = os.path.basename(f).replace('stocks_', '').replace('.csv', '')
    try:
        snap_df = pd.read_csv(f, index_col=0)
        if 'Final_Rank' not in snap_df.columns or 'Ticker' not in snap_df.columns:
            print(f'  Skipping {date_str}: missing required columns')
            continue
        picks = snap_df.sort_values('Final_Rank').head(TOP_N)['Ticker'].tolist()
        snapshots[date_str] = picks
    except Exception as e:
        print(f'  Skipping {date_str}: {e}')

snapshot_dates = sorted(snapshots.keys())
all_tickers = sorted(set(t for picks in snapshots.values() for t in picks))

print(f'Loaded {len(snapshots)} snapshots: {snapshot_dates[0]} to {snapshot_dates[-1]}')
print(f'Unique tickers across all snapshots: {len(all_tickers)}')
print(f'\nTop {TOP_N} picks per snapshot:')
for d, picks in sorted(snapshots.items()):
    print(f'  {d}: {", ".join(picks)}')

In [ ]:
## Cell 3 — BT-Prices: Batch-Download Adjusted Close Prices
# Downloads adjusted close prices (dividends + splits baked in) for every ticker
# that ever appeared in a top-N pick, plus SPY as the benchmark.
# One yf.download() call is far faster than looping per-ticker.
# Runtime: ~30-90 seconds depending on network speed.

fetch_start = (pd.Timestamp(snapshot_dates[0]) - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
fetch_end   = _dt.today().strftime('%Y-%m-%d')
tickers_to_fetch = all_tickers + ['SPY']

print(f'Downloading {len(tickers_to_fetch)} tickers  ({fetch_start} to {fetch_end}) ...')

raw = yf.download(tickers_to_fetch, start=fetch_start, end=fetch_end,
                  auto_adjust=True, progress=True)

# Extract the Close price layer (MultiIndex when >1 ticker)
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close'].copy()
else:
    prices = raw[['Close']].copy()
    prices.columns = tickers_to_fetch

# Normalise index to tz-naive dates so timestamp comparisons work cleanly
prices.index = pd.to_datetime(prices.index)
if prices.index.tz is not None:
    prices.index = prices.index.tz_localize(None)

found   = [t for t in tickers_to_fetch if t in prices.columns and prices[t].notna().any()]
missing = [t for t in tickers_to_fetch if t not in found]

print(f'\nSuccessfully fetched : {len(found)}/{len(tickers_to_fetch)} tickers')
if missing:
    print(f'Missing (delisted / unavailable): {", ".join(missing)}')
print(f'Price history shape  : {prices.shape[0]} trading days x {prices.shape[1]} tickers')

In [ ]:
## Cell 4 — BT-Returns: Compute Per-Vintage Returns vs SPY

def get_price(ticker, target_date, direction='forward', window_days=7):
    """Return nearest available adjusted close within window_days of target_date.
    direction='forward'  -> first price on or after target_date (entry price)
    direction='backward' -> last  price on or before target_date (exit price)
    """
    if ticker not in prices.columns:
        return None
    ts = pd.Timestamp(target_date)
    if direction == 'forward':
        mask = (prices.index >= ts) & (prices.index <= ts + pd.Timedelta(days=window_days))
        window = prices.loc[mask, ticker].dropna()
        return float(window.iloc[0]) if len(window) > 0 else None
    else:
        mask = (prices.index >= ts - pd.Timedelta(days=window_days)) & (prices.index <= ts)
        window = prices.loc[mask, ticker].dropna()
        return float(window.iloc[-1]) if len(window) > 0 else None

today_ts = pd.Timestamp(_dt.today().date())
bt_results = []

for snap_date in snapshot_dates:
    tickers = snapshots[snap_date]
    snap_ts = pd.Timestamp(snap_date)

    for label, months in HOLD_PERIODS.items():
        exit_ts = snap_ts + pd.DateOffset(months=months)

        # Skip if the exit date hasn't happened yet (no data to evaluate)
        if exit_ts > today_ts:
            continue

        # SPY benchmark return
        spy_entry = get_price('SPY', snap_date, direction='forward')
        spy_exit  = get_price('SPY', exit_ts.strftime('%Y-%m-%d'), direction='backward')
        if spy_entry is None or spy_exit is None:
            continue
        spy_ret = (spy_exit - spy_entry) / spy_entry * 100

        # Equal-weight portfolio return
        ticker_rets = {}
        for t in tickers:
            entry = get_price(t, snap_date, direction='forward')
            exit_ = get_price(t, exit_ts.strftime('%Y-%m-%d'), direction='backward')
            if entry is not None and exit_ is not None and entry > 0:
                ticker_rets[t] = (exit_ - entry) / entry * 100

        if len(ticker_rets) < MIN_VALID_TICKERS:
            print(f'  Skipping {snap_date} {label}: only {len(ticker_rets)} valid tickers')
            continue

        port_ret = float(np.mean(list(ticker_rets.values())))

        bt_results.append({
            'Snapshot Date':        snap_date,
            'Hold Period':          label,
            'Portfolio Return (%)': round(port_ret, 2),
            'SPY Return (%)':       round(spy_ret, 2),
            'Outperformance (%)':   round(port_ret - spy_ret, 2),
            'Valid Tickers':        len(ticker_rets),
            'Tickers':              ', '.join(ticker_rets.keys()),
        })

bt_df = pd.DataFrame(bt_results)

# ── Summary table ──────────────────────────────────────────────────────────────
period_order = [p for p in ['1M', '3M', '6M', '12M'] if p in bt_df['Hold Period'].unique()]

summary = (bt_df
    .groupby('Hold Period')
    .agg(
        Vintages       = ('Snapshot Date', 'count'),
        Portfolio_Avg  = ('Portfolio Return (%)', 'mean'),
        SPY_Avg        = ('SPY Return (%)', 'mean'),
        Outperf_Avg    = ('Outperformance (%)', 'mean'),
        Win_Rate_Pct   = ('Outperformance (%)', lambda x: round((x > 0).mean() * 100, 1)),
        Best_Vintage   = ('Outperformance (%)', 'max'),
        Worst_Vintage  = ('Outperformance (%)', 'min'),
    )
    .round(2)
    .reindex(period_order)
)

print('=' * 72)
print(f'  BACKTEST SUMMARY  |  Top-{TOP_N} Equal-Weight Portfolio vs SPY')
print('=' * 72)
print(summary.to_string())
print('=' * 72)
print(f'\nTotal vintage-period combinations evaluated: {len(bt_df)}')

In [ ]:
## Cell 5 — BT-Visualize: Charts

COLORS = {
    'portfolio': '#2196F3',   # blue
    'spy':       '#FF9800',   # orange
}

# ── Chart 1: Portfolio vs SPY returns, one grouped bar chart per hold period ───
for period in period_order:
    pdata = bt_df[bt_df['Hold Period'] == period].sort_values('Snapshot Date')
    if pdata.empty:
        continue

    win_rate   = (pdata['Outperformance (%)'] > 0).mean() * 100
    avg_outper = pdata['Outperformance (%)'].mean()

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=pdata['Snapshot Date'],
        y=pdata['Portfolio Return (%)'],
        name=f'Top-{TOP_N} Portfolio',
        marker_color=COLORS['portfolio'],
        text=pdata['Portfolio Return (%)'].map(lambda v: f'{v:+.1f}%'),
        textposition='outside',
    ))
    fig.add_trace(go.Bar(
        x=pdata['Snapshot Date'],
        y=pdata['SPY Return (%)'],
        name='SPY Benchmark',
        marker_color=COLORS['spy'],
        text=pdata['SPY Return (%)'].map(lambda v: f'{v:+.1f}%'),
        textposition='outside',
    ))
    fig.add_hline(y=0, line_color='black', line_width=0.8)
    fig.update_layout(
        title=f'{period} Returns — Top-{TOP_N} Portfolio vs SPY  |  Win Rate: {win_rate:.0f}%  |  Avg Outperformance: {avg_outper:+.2f}%',
        xaxis_title='Snapshot Date',
        yaxis_title='Return (%)',
        barmode='group',
        template='plotly_white',
        height=450,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()

# ── Chart 2: Outperformance heatmap (green = beat SPY, red = missed) ───────────
pivot = bt_df.pivot(index='Snapshot Date', columns='Hold Period', values='Outperformance (%)')
pivot = pivot.reindex(columns=[p for p in ['1M', '3M', '6M', '12M'] if p in pivot.columns])

text_vals = [
    [f'{v:+.1f}%' if (v == v) else 'N/A' for v in row]
    for row in pivot.values
]
fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=text_vals,
    texttemplate='%{text}',
    colorbar=dict(title='Outperf vs SPY (%)'),
))
fig.update_layout(
    title='Outperformance vs SPY — by Snapshot Date & Hold Period  (green = beat market)',
    xaxis_title='Hold Period',
    yaxis_title='Snapshot Date',
    template='plotly_white',
    height=max(420, len(pivot) * 30),
)
fig.show()

# ── Chart 3: Cumulative $10,000 growth using 1-month rebalancing ───────────────
one_m = bt_df[bt_df['Hold Period'] == '1M'].sort_values('Snapshot Date').reset_index(drop=True)

if len(one_m) >= 2:
    port_vals   = [10_000.0]
    spy_vals    = [10_000.0]
    chart_dates = [one_m['Snapshot Date'].iloc[0]]

    for _, row in one_m.iterrows():
        port_vals.append(port_vals[-1] * (1 + row['Portfolio Return (%)'] / 100))
        spy_vals.append(spy_vals[-1]   * (1 + row['SPY Return (%)']       / 100))
        chart_dates.append(row['Snapshot Date'])

    final_port    = port_vals[-1]
    final_spy     = spy_vals[-1]
    total_outperf = ((final_port - final_spy) / final_spy) * 100

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=chart_dates, y=port_vals,
        mode='lines+markers',
        name=f'Top-{TOP_N} Portfolio',
        line=dict(color=COLORS['portfolio'], width=2.5),
        hovertemplate='%{x}<br><b>$%{y:,.0f}</b><extra>Portfolio</extra>',
    ))
    fig.add_trace(go.Scatter(
        x=chart_dates, y=spy_vals,
        mode='lines+markers',
        name='SPY',
        line=dict(color=COLORS['spy'], width=2.5),
        hovertemplate='%{x}<br><b>$%{y:,.0f}</b><extra>SPY</extra>',
    ))
    fig.add_hline(y=10_000, line_dash='dot', line_color='grey', line_width=1,
                  annotation_text='Starting $10,000', annotation_position='top left')
    fig.update_layout(
        title=(f'$10,000 Compounded with Monthly Rebalancing  |  '
               f'Portfolio: ${final_port:,.0f}  vs  SPY: ${final_spy:,.0f}  '
               f'({total_outperf:+.1f}% edge)'),
        xaxis_title='Date',
        yaxis_title='Portfolio Value ($)',
        template='plotly_white',
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()
else:
    print('Not enough 1M vintages yet to draw a cumulative growth chart.')